# 🌿 Plant Disease Detection - Preprocessing
**Dataset:** PlantVillage
**Metode:** EfficientNetB4 + CBAM Attention Mechanism

In [ ]:
# Install dependencies
!pip install tensorflow pillow matplotlib seaborn scikit-learn -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API setup
# Upload kaggle.json terlebih dahulu
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset PlantVillage dari Kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/
!unzip -q /content/plantvillage-dataset.zip -d /content/plantvillage/

DATASET_PATH = '/content/plantvillage/plantvillage dataset/color'
print('Dataset path:', DATASET_PATH)

In [ ]:
# ── Eksplorasi Dataset ──
classes = sorted(os.listdir(DATASET_PATH))
classes = [c for c in classes if os.path.isdir(os.path.join(DATASET_PATH, c))]
print(f'Total kelas: {len(classes)}')

class_counts = {}
for c in classes:
    count = len(os.listdir(os.path.join(DATASET_PATH, c)))
    class_counts[c] = count
    print(f'  {c}: {count} gambar')

total = sum(class_counts.values())
print(f'\nTotal gambar: {total}')

In [ ]:
# ── Visualisasi Distribusi Kelas ──
plt.figure(figsize=(20, 6))
labels = list(class_counts.keys())
values = list(class_counts.values())
colors = ['#2e7d32' if 'healthy' in l.lower() else '#c62828' for l in labels]

bars = plt.bar(range(len(labels)), values, color=colors, edgecolor='white', linewidth=0.5)
plt.xticks(range(len(labels)), [l.split('___')[1][:15] for l in labels], rotation=90, fontsize=7)
plt.title('Distribusi Kelas Dataset PlantVillage', fontsize=14, fontweight='bold')
plt.xlabel('Kelas Penyakit')
plt.ylabel('Jumlah Gambar')

from matplotlib.patches import Patch
legend = [Patch(color='#2e7d32', label='Healthy'), Patch(color='#c62828', label='Disease')]
plt.legend(handles=legend)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/plant_disease/visualisasi/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distribusi dataset disimpan!')

In [ ]:
# ── Sample Gambar ──
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img
import random

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
sample_classes = random.sample(classes, min(24, len(classes)))

for ax, cls in zip(axes.flatten(), sample_classes):
    img_files = os.listdir(os.path.join(DATASET_PATH, cls))
    img_path = os.path.join(DATASET_PATH, cls, random.choice(img_files))
    img = load_img(img_path, target_size=(224, 224))
    ax.imshow(img)
    label = cls.split('___')[1].replace('_', ' ')[:20]
    color = '#1b5e20' if 'healthy' in cls.lower() else '#b71c1c'
    ax.set_title(label, fontsize=7, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sampel Gambar Dataset PlantVillage', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/plant_disease/visualisasi/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Data Generator dengan Augmentasi ──
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Augmentasi hanya pada training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    zoom_range=0.2,
    shear_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42
)
val_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42
)

print(f'Train: {train_gen.samples} gambar, {train_gen.num_classes} kelas')
print(f'Val:   {val_gen.samples} gambar')

# Simpan class indices
import json
os.makedirs('/content/drive/MyDrive/plant_disease', exist_ok=True)
with open('/content/drive/MyDrive/plant_disease/class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print('Class indices disimpan!')

In [ ]:
# ── Visualisasi Augmentasi ──
sample_img_path = os.path.join(DATASET_PATH, classes[0],
    os.listdir(os.path.join(DATASET_PATH, classes[0]))[0])
sample_img = load_img(sample_img_path, target_size=IMG_SIZE)
sample_arr = np.expand_dims(np.array(sample_img)/255.0, 0)

aug_gen = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    horizontal_flip=True, zoom_range=0.2
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0,0].imshow(sample_arr[0])
axes[0,0].set_title('Original', fontweight='bold')
axes[0,0].axis('off')

aug_it = aug_gen.flow(sample_arr, batch_size=1)
for i, ax in enumerate(axes.flatten()[1:]):
    aug_img = next(aug_it)[0]
    ax.imshow(aug_img)
    ax.set_title(f'Augmented {i+1}')
    ax.axis('off')

plt.suptitle('Contoh Augmentasi Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/plant_disease/visualisasi/augmentation_sample.png', dpi=150)
plt.show()
print('Preprocessing selesai! Generator siap digunakan untuk training.')